In [4]:
import pandas as pd
import torch
from torch_geometric.data import Data

# Load the merged and normalized data
df = pd.read_csv("../../dataset/final_normalized_graph.csv")

unique_nodes = pd.concat([df['source_number'], df['destination_number']]).unique()
node_mapping = {old_id: new_id for new_id, old_id in enumerate(sorted(unique_nodes))}

# Apply mapping
src = df['source_number'].map(node_mapping).values
dst = df['destination_number'].map(node_mapping).values

edge_index = torch.tensor([src, dst], dtype=torch.long)

# Notice we stripped out actual_time and actual_distance to prevent target leakage
feature_cols = [
    'is_carting', 
    'is_ftl', 
    'day_of_week', 
    'start_hour', 
    'osrm_time', 
    'osrm_distance'
]
edge_attr = torch.tensor(df[feature_cols].values, dtype=torch.float32)

# Create y (The target to predict)
# Shape: [E, 1]
target_col = ['real_actual_time_factor']
y = torch.tensor(df[target_col].values, dtype=torch.float32)

#  Build the PyTorch Geometric Data object
train_data = Data(edge_index=edge_index, edge_attr=edge_attr, y=y)

print(f"Graph created with {train_data.num_nodes} nodes and {train_data.num_edges} edges.")
print(f"Edge feature dimensions: {train_data.edge_attr.shape[1]}")

/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Graph created with 1562 nodes and 15695 edges.
Edge feature dimensions: 6


/tmp/ipykernel_9697/693887280.py:15: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  edge_index = torch.tensor([src, dst], dtype=torch.long)
/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/torch_geometric/data/data.py:187: UserWarning: Unable to accurately infer 'num_nodes' from the attribute set '{'edge_index', 'y', 'edge_attr'}'. Please explicitly set 'num_nodes' as an attribute of 'data' to suppress this warning
  return sum([v.num_nodes for v in self.node_stores])


In [5]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import CGConv

class DelhiveryGNN(nn.Module):
    def __init__(self, num_nodes, embed_dim=16, num_edge_features=6):
        super(DelhiveryGNN, self).__init__()
        
        #  Node Embeddings
        self.node_emb = nn.Embedding(num_nodes, embed_dim)
        
        # Message passing layer (has 6 edge features)
        self.conv1 = CGConv(channels=embed_dim, dim=num_edge_features)
        
        # 3. Final Prediction Head
        # Input size: Source Node (16) + Dest Node (16) + Edge Features (6) = 38
        predictor_input_size = (embed_dim * 2) + num_edge_features
        self.predictor = nn.Sequential(
            nn.Linear(predictor_input_size, 32),
            nn.ReLU(),
            nn.Dropout(p=0.4),   # dropout to handle big error
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1) 
        )

    def forward(self, edge_index, edge_attr):
        # Generate the initial node vectors
        num_nodes = self.node_emb.num_embeddings
        node_ids = torch.arange(num_nodes, device=edge_index.device)
        x = self.node_emb(node_ids)
        
        # Message Passing (Nodes learn from their neighbors and the edge features)
        x = self.conv1(x, edge_index, edge_attr)
        x = F.relu(x)
        
        # Extract the updated source and destination nodes for every edge
        source_nodes = x[edge_index[0]] 
        dest_nodes = x[edge_index[1]]   
        
        # Concatenate nodes with edge features to make the final prediction
        regression_input = torch.cat([source_nodes, dest_nodes, edge_attr], dim=1)
        predictions = self.predictor(regression_input)
        
        return predictions

In [6]:
# 1. Initialize the Model
# train_data.num_nodes ensures we create exactly enough embeddings for all hubs

model = DelhiveryGNN(
    num_nodes=train_data.num_nodes, 
    embed_dim=16, 
    num_edge_features=train_data.edge_attr.shape[1]
)

# 2. Setup the Optimizer and Loss Function
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.SmoothL1Loss()

# 3. Execute the Epochs
epochs = 200
print("Starting Training...")

model.train() # Set model to training mode
for epoch in range(epochs):
    # Clear old gradients
    optimizer.zero_grad()
    
    # Forward Pass: Make predictions
    predictions = model(train_data.edge_index, train_data.edge_attr)
    
    # Calculate Error (MSE between predictions and real target)
    loss = criterion(predictions, train_data.y)
    
    # Backpropagation: Calculate the derivatives
    loss.backward()
    
    # Optimizer Step: Update the embeddings and weights
    optimizer.step()
    
    # Print progress every 20 epochs
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:03d}/{epochs} | Loss: {loss.item():.4f}")
    # print(f"loss = {loss.item():.4f}  ::::: epochs = {epochs}, lr = {lr}")
# print("Training Complete!")

Starting Training...
Epoch 020/200 | Loss: 1.0113
Epoch 040/200 | Loss: 0.7387
Epoch 060/200 | Loss: 0.6273
Epoch 080/200 | Loss: 0.5717
Epoch 100/200 | Loss: 0.5274
Epoch 120/200 | Loss: 0.4998
Epoch 140/200 | Loss: 0.4807
Epoch 160/200 | Loss: 0.4644
Epoch 180/200 | Loss: 0.4468
Epoch 200/200 | Loss: 0.4363


In [7]:
# train(2000, 0.02)
# train(3000, 0.03)
# train(4000, 0.05)
# train(2000, 0.03)
# train(500, 0.05)
# train(1000, 0.03)

loss, is too much. switching from nn.MSELoss to nn.SmoothL1Loss

In [8]:
# Save the model state dictionary (the weights and the learned embeddings)
torch.save(model.state_dict(), "../../final_model/delhivery_gnn_weights.pth")
print("Model weights saved to 'delhivery_gnn_weights.pth'")

Model weights saved to 'delhivery_gnn_weights.pth'


## Testing on test_data

In [9]:
import pandas as pd
import torch
import torch.nn as nn
from torch_geometric.data import Data
import joblib

# Load the fitted scale that was used on the training data
scaler = joblib.load('../../dataset/fitted_scaler.pkl')

# Preprocessing the test data, same as train data to match input of our model
# ==========================================
test_df = pd.read_csv("../../dataset/test_data.csv")
test_df = test_df.drop(columns=['is_day', 'is_night', 'is_cutoff'], errors='ignore')

grouping_columns = ['source_number', 'destination_number', 'is_carting', 'is_ftl', 'day_of_week', 'start_hour']
aggregation_logic = {
    'osrm_time': 'median',
    'osrm_distance': 'median',
    'actual_distance': 'median',
    'actual_time': 'median',
    'real_actual_time_factor': 'median'
}
# Create the structural test graph
merged_test_df = test_df.groupby(grouping_columns).agg(aggregation_logic).reset_index()


# Apply scaling, and node mapping
# ==========================================
continuous_cols = ['osrm_time', 'osrm_distance', 'actual_distance', 'actual_time']
merged_test_df[continuous_cols] = scaler.transform(merged_test_df[continuous_cols])

# Round off
columns_to_round = continuous_cols + ['real_actual_time_factor']
merged_test_df[columns_to_round] = merged_test_df[columns_to_round].round(4)

merged_test_df['src_mapped'] = merged_test_df['source_number'].map(node_mapping)
merged_test_df['dst_mapped'] = merged_test_df['destination_number'].map(node_mapping)

# Drop any trips involving brand-new hubs that the training model has never seen
merged_test_df = merged_test_df.dropna(subset=['src_mapped', 'dst_mapped'])

# Save the clean test graph just to have it
merged_test_df.to_csv("../../dataset/final_normalized_test_graph.csv", index=False)
print("Test data normalization and mapping complete!")

# CONVERT TO PYTORCH TENSORS
# ==========================================
edge_index = torch.tensor([
    merged_test_df['src_mapped'].astype(int).values, 
    merged_test_df['dst_mapped'].astype(int).values
], dtype=torch.long)

feature_cols = ['is_carting', 'is_ftl', 'day_of_week', 'start_hour', 'osrm_time', 'osrm_distance']
edge_attr = torch.tensor(merged_test_df[feature_cols].values, dtype=torch.float32)

y = torch.tensor(merged_test_df['real_actual_time_factor'].values, dtype=torch.float32).view(-1, 1)

test_data = Data(edge_index=edge_index, edge_attr=edge_attr, y=y)
print(f"Test Graph created with {test_data.num_edges} edges.")

# EVALUATE THE MODEL
# ==========================================
model = DelhiveryGNN(num_nodes=len(node_mapping), embed_dim=16, num_edge_features=6)
model.load_state_dict(torch.load("../../final_model/delhivery_gnn_weights.pth"))

model.eval()
criterion = nn.SmoothL1Loss() 

with torch.no_grad():
    test_predictions = model(test_data.edge_index, test_data.edge_attr)
    test_loss = criterion(test_predictions, test_data.y)

print(f"Final Test Loss: {test_loss.item():.4f}")

Test data normalization and mapping complete!
Test Graph created with 7047 edges.
Final Test Loss: 0.7003


In [10]:
import torch
import numpy as np

def test_single_route(source_hub_raw, dest_hub_raw, is_carting, is_ftl, day, hour, raw_osrm_time, raw_osrm_distance):
    print(f"--- PREDICTING ETA: Hub {source_hub_raw} -> Hub {dest_hub_raw} ---")
    
    # Map the raw hub numbers to the AI's internal IDs
    if source_hub_raw not in node_mapping or dest_hub_raw not in node_mapping:
        return "ERROR: One of these hubs is brand new and wasn't in the training data!"
    
    src_id = torch.tensor([node_mapping[source_hub_raw]], dtype=torch.long)
    dst_id = torch.tensor([node_mapping[dest_hub_raw]], dtype=torch.long)
    
    #  Scale the continuous features using the saved scaler
    # We must pass 4 columns to avoid an error, so we just pass 0s for the actuals (the AI ignores them anyway).
    dummy_input = np.array([[raw_osrm_time, raw_osrm_distance, 0.0, 0.0]])
    scaled_values = scaler.transform(dummy_input)[0]
    
    scaled_time = scaled_values[0]
    scaled_dist = scaled_values[1]
    
    # Create the final 6-feature vector
    # Format: ['is_carting', 'is_ftl', 'day_of_week', 'start_hour', 'osrm_time', 'osrm_distance']
    edge_attr = torch.tensor([[is_carting, is_ftl, day, hour, scaled_time, scaled_dist]], dtype=torch.float32)
    
    # Evaluate it through the frozen model
    model.load_state_dict(torch.load("../../final_model/delhivery_gnn_weights.pth"))
    model.eval()
    with torch.no_grad():
        x_src = model.node_emb(src_id)
        x_dst = model.node_emb(dst_id)
        
        regression_input = torch.cat([x_src, x_dst, edge_attr], dim=1)
        
        predicted_factor = model.predictor(regression_input).item()
        
    # 5. Calculate the True ETA
    predicted_factor = min(predicted_factor, 4.0)
    true_eta = raw_osrm_time * predicted_factor
    
    print(f"OSRM Base Time:       {raw_osrm_time} hours")
    print(f"AI Predicted Factor:  {predicted_factor:.2f}x")
    print(f"FINAL TRUE ETA:       {true_eta:.2f} hours")
    print("-" * 50)
    
    return true_eta


testing_str = "test,444,30,1,0,3,23,0,1,0,52.0,56.2137,45.90293875782862,68.0,1.3076923076923077";
[hub_A, hub_B, is_carting, is_ftl, day_of_week, start_hour, is_day, is_night, is_cutoff, osrm_time, osrm_distance,actual_distance,actual_time,real_actual_time_factor] = map(lambda x: float(x), testing_str.split(',')[1:]);

# BARE MODEL TESTING FOR SELF SATISFRACTION
# ==========================================

# Pick two valid hubs from your dataset (replace 0 and 1 with actual hubs if needed)
# hub_A = 4
# hub_B = 39

test_single_route(
    source_hub_raw=hub_A, 
    dest_hub_raw=hub_B, 
    is_carting=is_carting, is_ftl=is_ftl, 
    day=day_of_week, hour=start_hour, 
    raw_osrm_time=osrm_time, raw_osrm_distance=osrm_distance
)
print(actual_time)

# Scenario 2: An FTL truck on Friday (Day 4) at 11:00 PM (Night traffic)
# print("\n[Scenario 2: FTL | Friday Night]")
# test_single_route(
#     source_hub_raw=hub_A, 
#     dest_hub_raw=hub_B, 
#     is_carting=0, is_ftl=1, 
#     day=4, hour=23, 
#     raw_osrm_time=12.5, raw_osrm_distance=400.0
# )

--- PREDICTING ETA: Hub 444.0 -> Hub 30.0 ---
OSRM Base Time:       52.0 hours
AI Predicted Factor:  2.85x
FINAL TRUE ETA:       148.03 hours
--------------------------------------------------
68.0


/home/swan/Desktop/projects/ETA-pred-IITG/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [11]:
import joblib
joblib.dump(node_mapping, '../../dataset/node_mapping.pkl')

['../../dataset/node_mapping.pkl']